# 03 — Topic modelling (LDA)

Cluster the residual (non-government, non-candidate) ad bodies into latent topics so we can:

- Filter out commercial/spam topics (fashion, fitness apps, retail) from the political-adjacent subset.
- Surface what *kinds* of political messaging are circulating outside party/candidate channels — climate, cost-of-living, Voice, housing, etc.
- Layer topic labels into the v3 parquet for cross-tabbing with sentiment (notebook 04) and spend/impressions.

Approach: fit a single LDA model on the full residual corpus, eyeball the top words per topic, hand-label each topic in a CSV, join the labels back to the corpus.

## 1. Preprocessing

Load v2 parquet, filter to one row per ad (`ad_seq_no = 1`) and non-classified (`match_type IS NULL`). Extract the first creative body, tokenise with `RegexTokenizer`, drop stop words (English defaults + domain-specific noise). Vectorise word counts with `CountVectorizer` — raw counts, not TF-IDF, since LDA expects integer term frequencies.

Cache the vectorised features so subsequent LDA fits (at different `k`) don't re-run preprocessing.

## 2. Explore `k` on a sample

Fit LDA at several `k` values (5, 10, 15, 20) on a 10% sample of the cached features. Print top-12 words per topic for each `k`. Eyeball the printouts to pick a `k` where topics are distinct and each list reads as a coherent theme.

Fix `seed=42` so comparing `k=10` vs `k=15` isn't muddled by random init differences. Cheap and disposable — no parquet writes from this section.

## 3. Final fit on full corpus

Refit LDA at the chosen `k` on the full cached features. Transform the corpus to attach `topicDistribution` (length-`k` vector) and `topic_id` (argmax) to every ad. Persist:

- **Intermediate parquet** — corpus + `topicDistribution` + `topic_id`. Expensive to recompute, so write it once.
- **`data/topic_terms.csv`** — small file, one row per topic, top-15 most representative terms each.

Optionally save the fitted `PipelineModel` so the same topic model can be applied to new ads later.

## 4. Manual labelling (out-of-notebook)

Open `data/topic_terms.csv` in a spreadsheet, add a `label` column with human-readable names (e.g. `climate`, `cost_of_living`, `voice_referendum`, `commercial_retail`, `noise`). Save as `data/topic_labels.csv`.

Topics that look like commercial noise (fashion brands, fitness app keywords, etc.) get labels like `commercial_*` or `noise` — these are the categories notebook 04 / later filters will drop.

## 5. Join labels back

Read intermediate parquet + `data/topic_labels.csv`. Broadcast-join on `topic_id` to add a `topic_label` column. Write the result as v3 parquet — same schema as v2 with `topicDistribution`, `topic_id`, and `topic_label` appended.

Fully re-runnable: tweak labels, re-run this section, no LDA refit needed.